# Notebook 8: Inference & Generation

## Learning Objectives
- Implement greedy decoding
- Implement beam search from scratch
- Implement temperature, top-k, top-p sampling
- Understand model quantization

## Task 1: Greedy Decoding

### HINT: Always pick the highest probability token
```python
next_token = argmax(logits)
```

Simple but can produce repetitive text.

In [ ]:
import torch
import torch.nn.functional as F

def greedy_decode(model, input_ids, max_new_tokens, eos_token_id=2):
    """Greedy decoding - always pick highest probability token"""
    model.eval()
    generated = input_ids.clone()
    
    with torch.no_grad():
        for _ in range(max_new_tokens):
            # Forward pass
            logits = model(generated)
            
            # Get last token logits
            next_token_logits = logits[:, -1, :]
            
            # Greedy: argmax
            next_token = next_token_logits.argmax(dim=-1, keepdim=True)
            
            generated = torch.cat([generated, next_token], dim=1)
            
            # Stop if EOS
            if next_token.item() == eos_token_id:
                break
    
    return generated

# Test greedy
from notebook_06_block import DeepSeekV3

config = {
    'vocab_size': 1000,  # Small for demo
    'hidden_size': 128,
    'num_layers': 2,
    'num_heads': 4,
    'num_kv_heads': 1,
    'num_experts': 2,
    'num_active_experts': 2,
    'intermediate_size': 256
}

model = DeepSeekV3(config)
input_ids = torch.tensor([[1, 2, 3, 4, 5]])

output = greedy_decode(model, input_ids, max_new_tokens=10)
print(f"Input:  {input_ids.tolist()}")
print(f"Output: {output.tolist()}")

## Task 2: Beam Search

### HINT: Keep top-k paths
```python
beam_width = 3
for step:
    for each beam:
        get top-k next tokens
    keep best beam_width sequences
```

Better than greedy - explores multiple paths!

In [ ]:
def beam_search(model, input_ids, beam_width=3, max_new_tokens=10, eos_token_id=2):
    """Beam search - keep top beam_width sequences"""
    model.eval()
    
    with torch.no_grad():
        # Initialize: [beam_width, seq_len]
        sequences = [input_ids]
        scores = [0.0]
        
        for step in range(max_new_tokens):
            candidates = []
            
            for seq, score in zip(sequences, scores):
                # Forward pass
                logits = model(seq)
                
                # Get last token logits
                next_logits = logits[:, -1, :]
                
                # Get top-k tokens
                topk_logits, topk_ids = torch.topk(next_logits, beam_width, dim=-1)
                
                # Add log probabilities
                log_probs = F.log_softmax(next_logits, dim=-1)
                topk_log_probs = torch.gather(log_probs, -1, topk_ids)
                
                # Create new candidates
                for i in range(beam_width):
                    new_seq = torch.cat([seq, topk_ids[:, i:i+1]], dim=1)
                    new_score = score + topk_log_probs[:, i].item()
                    candidates.append((new_seq, new_score))
            
            # Sort by score and keep top beam_width
            candidates.sort(key=lambda x: x[1], reverse=True)
            sequences = [c[0] for c in candidates[:beam_width]]
            scores = [c[1] for c in candidates[:beam_width]]
            
            # Check for EOS
            if all(seq[0, -1].item() == eos_token_id for seq in sequences):
                break
        
        # Return best sequence
        best_idx = scores.index(max(scores))
        return sequences[best_idx]

# Test beam search
output = beam_search(model, input_ids, beam_width=3, max_new_tokens=10)
print(f"Beam search output: {output.tolist()}")

## Task 3: Temperature Sampling

### HINT: Control randomness
```python
# temperature > 1: more random
# temperature < 1: more deterministic
# temperature = 0: equivalent to greedy

probs = softmax(logits / temperature)
next_token = multinomial(probs)
```

In [ ]:
def temperature_sample(logits, temperature=1.0):
    """Sample from logits using temperature"""
    if temperature == 0:
        return logits.argmax(dim=-1)
    
    # Apply temperature
    logits = logits / temperature
    
    # Softmax
    probs = F.softmax(logits, dim=-1)
    
    # Sample
    return torch.multinomial(probs, 1)

# Test different temperatures
logits = torch.randn(1, 1000)  # Random logits

print("=== Temperature Effects ===")
for temp in [0.1, 0.5, 1.0, 2.0]:
    samples = [temperature_sample(logits, temp).item() for _ in range(10)]
    print(f"Temp={temp}: {samples[:5]}...")

## Task 4: Top-K and Top-P (Nucleus) Sampling

### HINT:
```
Top-K: Only consider top-k tokens

Top-P: Only consider smallest set with cumulative prob > p
```

In [ ]:
def top_k_sample(logits, k=50):
    """Top-K sampling - only consider top k tokens"""
    topk_logits, topk_indices = torch.topk(logits, k)
    probs = F.softmax(topk_logits, dim=-1)
    sampled = torch.multinomial(probs, 1)
    return topk_indices.gather(-1, sampled)

def top_p_sample(logits, p=0.9):
    """Nucleus (Top-P) sampling - smallest set with prob > p"""
    probs = F.softmax(logits, dim=-1)
    sorted_probs, sorted_indices = torch.sort(probs, descending=True)
    
    # Cumulative probability
    cumsum = torch.cumsum(sorted_probs, dim=-1)
    
    # Mask tokens after threshold
    mask = cumsum > p
    mask[..., 1:] = mask[..., :-1].clone()
    mask[..., 0] = False
    
    # Zero out probabilities
    sorted_probs[mask] = 0
    
    # Renormalize
    sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
    
    # Sample
    sampled = torch.multinomial(sorted_probs, 1)
    return sorted_indices.gather(-1, sampled)

# Test
logits = torch.randn(1, 1000)

print("Top-10:", top_k_sample(logits, k=10).item())
print("Top-P=0.9:", top_p_sample(logits, p=0.9).item())

## Task 5: Complete Generation Function

### HINT: Putting it all together

In [ ]:
def generate(
    model,
    input_ids,
    max_new_tokens,
    method='greedy',  # greedy, beam, temp, topk, topp
    **kwargs
):
    """Complete generation function with multiple methods"""
    model.eval()
    generated = input_ids.clone()
    
    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits = model(generated)
            next_logits = logits[:, -1, :]
            
            if method == 'greedy':
                next_token = next_logits.argmax(dim=-1, keepdim=True)
            elif method == 'beam':
                return beam_search(model, input_ids, kwargs.get('beam_width', 3), max_new_tokens)
            elif method == 'temp':
                next_token = temperature_sample(next_logits, kwargs.get('temperature', 1.0))
            elif method == 'topk':
                next_token = top_k_sample(next_logits, kwargs.get('k', 50))
            elif method == 'topp':
                next_token = top_p_sample(next_logits, kwargs.get('p', 0.9))
            else:
                raise ValueError(f"Unknown method: {method}")
            
            generated = torch.cat([generated, next_token], dim=1)
    
    return generated

# Compare methods
print("=== Generation Comparison ===")
for method in ['greedy', 'temp', 'topk', 'topp']:
    output = generate(model, input_ids, 10, method=method)
    print(f"{method}: {output[0, :5].tolist()}...")

## Verification

Complete these checks:
1. ✅ Can implement greedy decoding
2. ✅ Can implement beam search from scratch
3. ✅ Can implement temperature, top-k, top-p sampling
4. ✅ Understand when to use each method